# Senior Data Engineer 技能清单 - Spark 篇

本笔记本帮助你检验和巩固成为 Senior DE 所需的 Spark 核心技能。

## 技能等级说明
- 🟢 **必须精通** - 面试必考，日常高频使用
- 🟡 **需要熟练** - 经常用到，需要能快速写出
- 🔵 **了解即可** - 知道概念，用时能查文档

---

# Part 1: Spark 核心概念

这些概念面试必问，必须能清晰解释。

## 1.1 🟢 SparkSession 配置

理解关键配置参数及其影响。

In [ ]:
from pyspark.sql import SparkSession

# ===== 生产级别的 SparkSession 配置 =====
spark = SparkSession.builder \
    .appName("Senior DE Spark Practice") \
    .master("local[*]") \
    \
    # 内存配置
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    \
    # Shuffle 配置 (重要!)
    .config("spark.sql.shuffle.partitions", "200") \
    \
    # 自适应查询执行 (Spark 3.0+)
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    \
    # 广播 Join 阈值
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")

In [ ]:
# 加载数据
df = spark.read.parquet("/workspace/data/nyc_taxi_2023_01.parquet")
print(f"加载了 {df.count():,} 条记录")
print(f"分区数: {df.rdd.getNumPartitions()}")

## 1.2 🟢 Transformations vs Actions (惰性求值)

**面试必问！** 必须能解释清楚。

In [ ]:
import time

# ===== Transformation: 惰性，不会立即执行 =====
print("=== Transformations (不触发计算) ===")
start = time.time()

# 这些都是 Transformation，不会触发计算
df_filtered = df.filter(df.total_amount > 50)  # filter
df_selected = df_filtered.select("tpep_pickup_datetime", "total_amount")  # select
df_sorted = df_selected.orderBy("total_amount", ascending=False)  # orderBy

print(f"Transformation 完成: {time.time() - start:.3f}s (几乎瞬间)")
print(f"类型: {type(df_sorted)}")

In [ ]:
# ===== Action: 触发实际计算 =====
print("\n=== Actions (触发计算) ===")
start = time.time()

# 这些是 Action，会触发计算
result = df_sorted.limit(10).collect()  # collect 是 Action

print(f"Action 完成: {time.time() - start:.3f}s")
print(f"返回类型: {type(result)}")
print(f"结果条数: {len(result)}")

In [ ]:
# ===== 常见 Transformations vs Actions =====
"""
Transformations (惰性):          Actions (立即执行):
- select()                       - collect()
- filter() / where()             - count()
- groupBy()                      - show()
- orderBy() / sort()             - first() / head()
- join()                         - take(n)
- withColumn()                   - write.xxx()
- drop()                         - foreach()
- distinct()                     - reduce()
- union()                        - toPandas()
- repartition()                  - saveAsTable()
- coalesce()
"""
print("见上方注释")

## 1.3 🟢 Spark 执行计划 (Explain)

调优必备，能看懂执行计划。

In [ ]:
# ===== 查看执行计划 =====
df.createOrReplaceTempView("trips")

query = spark.sql("""
    SELECT 
        PULocationID,
        COUNT(*) as trip_count,
        AVG(total_amount) as avg_fare
    FROM trips
    WHERE total_amount > 10
    GROUP BY PULocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")

# 简单执行计划
query.explain()

In [ ]:
# 详细执行计划 (包含物理计划)
query.explain(mode="extended")

In [ ]:
# ===== 执行计划关键点 =====
"""
需要关注的信息:

1. Scan: 数据读取
   - FileScan parquet: 读取 parquet 文件
   - 注意 PushedFilters: 谓词下推 (好事!)

2. Exchange: Shuffle 操作 (昂贵!)
   - hashpartitioning: 按哈希重分区
   - SinglePartition: 收集到单节点
   - 尽量减少 Shuffle!

3. HashAggregate: 聚合操作
   - 通常会看到两次: partial + final

4. BroadcastHashJoin vs SortMergeJoin:
   - Broadcast: 小表广播，效率高
   - SortMerge: 大表 Join，需要 Shuffle

5. Project: 列选择

6. Filter: 过滤操作
"""
print("见上方注释")

---

# Part 2: DataFrame API 精通

Senior DE 必须能熟练使用 DataFrame API。

## 2.1 🟢 基础操作

In [ ]:
from pyspark.sql.functions import col, lit, when, coalesce
from pyspark.sql.types import StringType, IntegerType, DoubleType

# ===== select: 多种写法 =====
# 方式1: 字符串
df.select("tpep_pickup_datetime", "total_amount").show(2)

# 方式2: col() 函数
df.select(col("tpep_pickup_datetime"), col("total_amount")).show(2)

# 方式3: df.column
df.select(df.tpep_pickup_datetime, df.total_amount).show(2)

# 方式4: 带表达式
df.select(
    col("total_amount"),
    (col("total_amount") * 1.1).alias("with_tax")
).show(2)

In [ ]:
# ===== filter / where (两者等价) =====
# 字符串表达式
df.filter("total_amount > 50 AND trip_distance > 10").count()

# Column 表达式 (推荐: 类型安全)
df.filter((col("total_amount") > 50) & (col("trip_distance") > 10)).count()

# where 是 filter 的别名
df.where(col("payment_type") == 1).count()

In [ ]:
# ===== withColumn: 添加/修改列 =====
df_new = df \
    .withColumn("fare_with_tax", col("total_amount") * 1.08) \
    .withColumn("is_long_trip", when(col("trip_distance") > 10, True).otherwise(False)) \
    .withColumn("payment_label", 
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .otherwise("Other")
    )

df_new.select("total_amount", "fare_with_tax", "trip_distance", "is_long_trip", "payment_label").show(5)

In [ ]:
# ===== withColumnRenamed / alias =====
# 重命名列
df.withColumnRenamed("tpep_pickup_datetime", "pickup_time").columns[:5]

In [ ]:
# ===== drop: 删除列 =====
df.drop("VendorID", "store_and_fwd_flag").columns

In [ ]:
# ===== distinct / dropDuplicates =====
# 所有列去重
df.select("payment_type").distinct().show()

# 按指定列去重 (保留第一条)
df.dropDuplicates(["PULocationID", "payment_type"]).count()

## 2.2 🟢 聚合操作

In [ ]:
from pyspark.sql.functions import count, sum, avg, min, max, round, countDistinct, collect_list, collect_set

# ===== 基础聚合 =====
df.agg(
    count("*").alias("total_trips"),
    countDistinct("PULocationID").alias("unique_pickup_locations"),
    round(avg("total_amount"), 2).alias("avg_fare"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    min("tpep_pickup_datetime").alias("first_trip"),
    max("tpep_pickup_datetime").alias("last_trip")
).show(truncate=False)

In [ ]:
# ===== groupBy 聚合 =====
df.groupBy("payment_type") \
    .agg(
        count("*").alias("trip_count"),
        round(avg("total_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    ) \
    .orderBy(col("trip_count").desc()) \
    .show()

In [ ]:
# ===== 多列 groupBy =====
from pyspark.sql.functions import date_format, hour

df.groupBy(
    date_format("tpep_pickup_datetime", "yyyy-MM-dd").alias("date"),
    hour("tpep_pickup_datetime").alias("hour")
).agg(
    count("*").alias("trips"),
    round(sum("total_amount"), 2).alias("revenue")
).orderBy("date", "hour").show(10)

In [ ]:
# ===== pivot: 数据透视 =====
# 每天各支付方式的行程数
df.groupBy(date_format("tpep_pickup_datetime", "yyyy-MM-dd").alias("date")) \
    .pivot("payment_type", [1, 2, 3, 4]) \
    .agg(count("*")) \
    .orderBy("date") \
    .show(5)

## 2.3 🟢 窗口函数 (Window Functions)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, ntile

# ===== 定义窗口 =====
# 按支付方式分区，按金额降序
window_by_payment = Window.partitionBy("payment_type").orderBy(col("total_amount").desc())

# 按日期排序的全局窗口
window_by_date = Window.orderBy("tpep_pickup_datetime")

In [ ]:
# ===== ROW_NUMBER / RANK / DENSE_RANK =====
df_ranked = df \
    .filter(col("payment_type").isin([1, 2])) \
    .withColumn("row_num", row_number().over(window_by_payment)) \
    .withColumn("rank", rank().over(window_by_payment)) \
    .withColumn("dense_rank", dense_rank().over(window_by_payment))

# 每种支付方式的 Top 3
df_ranked.filter(col("row_num") <= 3) \
    .select("payment_type", "total_amount", "row_num", "rank", "dense_rank") \
    .orderBy("payment_type", "row_num") \
    .show(6)

In [ ]:
# ===== LAG / LEAD: 前后行值 =====
from pyspark.sql.functions import to_date

# 每日统计
daily_stats = df.groupBy(to_date("tpep_pickup_datetime").alias("date")) \
    .agg(count("*").alias("trips"), round(sum("total_amount"), 2).alias("revenue"))

window_date = Window.orderBy("date")

daily_stats \
    .withColumn("prev_day_trips", lag("trips", 1).over(window_date)) \
    .withColumn("next_day_trips", lead("trips", 1).over(window_date)) \
    .withColumn("trip_change", col("trips") - lag("trips", 1).over(window_date)) \
    .orderBy("date") \
    .show(10)

In [ ]:
# ===== 累计求和 / 移动平均 =====
from pyspark.sql.functions import sum as _sum, avg as _avg

# 累计窗口: 从开始到当前行
cumulative_window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 移动窗口: 前6行到当前行 (7天移动平均)
moving_window = Window.orderBy("date").rowsBetween(-6, 0)

daily_stats \
    .withColumn("cumulative_trips", _sum("trips").over(cumulative_window)) \
    .withColumn("moving_avg_7d", round(_avg("trips").over(moving_window), 0)) \
    .orderBy("date") \
    .show(10)

In [ ]:
# ===== 🔥 练习 1: 窗口函数 =====

# 需求: 对于每个上车地点(PULocationID)，计算:
# 1. 该地点的总行程数
# 2. 该地点的行程数在所有地点中的排名
# 3. 该地点收入占总收入的百分比

# 你的代码:


## 2.4 🟢 Join 操作

In [ ]:
# 创建参考表
from pyspark.sql import Row

payment_types = spark.createDataFrame([
    Row(payment_type=1, payment_name="Credit Card"),
    Row(payment_type=2, payment_name="Cash"),
    Row(payment_type=3, payment_name="No Charge"),
    Row(payment_type=4, payment_name="Dispute"),
    Row(payment_type=5, payment_name="Unknown"),  # 这个在 trips 中没有
])

locations = spark.createDataFrame([
    Row(location_id=132, zone_name="JFK Airport"),
    Row(location_id=138, zone_name="LaGuardia Airport"),
    Row(location_id=161, zone_name="Midtown Center"),
])

In [ ]:
# ===== Join 类型 =====

# Inner Join (默认): 只保留两边都有的
df.join(payment_types, "payment_type", "inner") \
    .groupBy("payment_name").count().show()

# Left Join: 保留左表所有行
df.join(locations, df.PULocationID == locations.location_id, "left") \
    .filter(col("zone_name").isNotNull()) \
    .groupBy("zone_name").count().show()

# Left Anti: 左表中不在右表的行 (类似 NOT IN)
df.join(locations, df.PULocationID == locations.location_id, "left_anti") \
    .select("PULocationID").distinct().count()

In [ ]:
# ===== Broadcast Join: 小表广播 =====
from pyspark.sql.functions import broadcast

# 显式广播小表，避免 Shuffle
df.join(broadcast(payment_types), "payment_type") \
    .groupBy("payment_name").count().explain()

In [ ]:
# ===== 处理 Join 后的重复列名 =====
# 问题: 两个表有同名列
df1 = spark.createDataFrame([(1, "A"), (2, "B")], ["id", "value"])
df2 = spark.createDataFrame([(1, "X"), (2, "Y")], ["id", "value"])

# 方式1: 使用别名
df1.alias("d1").join(df2.alias("d2"), col("d1.id") == col("d2.id")) \
    .select(col("d1.id"), col("d1.value").alias("value1"), col("d2.value").alias("value2")).show()

# 方式2: Join 前重命名
df2_renamed = df2.withColumnRenamed("value", "value2")
df1.join(df2_renamed, "id").show()

## 2.5 🟡 数据类型与转换

In [ ]:
from pyspark.sql.functions import to_date, to_timestamp, date_format, unix_timestamp
from pyspark.sql.functions import cast

# ===== 类型转换 =====
df.select(
    col("total_amount"),
    col("total_amount").cast("int").alias("as_int"),
    col("total_amount").cast("string").alias("as_string"),
    col("passenger_count").cast(DoubleType()).alias("as_double")
).show(3)

In [ ]:
# ===== 日期时间处理 =====
df.select(
    col("tpep_pickup_datetime"),
    to_date("tpep_pickup_datetime").alias("date_only"),
    date_format("tpep_pickup_datetime", "yyyy-MM").alias("year_month"),
    date_format("tpep_pickup_datetime", "EEEE").alias("day_name"),
    unix_timestamp("tpep_pickup_datetime").alias("unix_ts")
).show(3, truncate=False)

In [ ]:
# ===== NULL 处理 =====
from pyspark.sql.functions import isnan, isnull, coalesce, nullif

# 检查 NULL
df.filter(col("passenger_count").isNull()).count()

# 填充 NULL
df.fillna({"passenger_count": 1, "tip_amount": 0})

# coalesce: 返回第一个非空值
df.select(
    col("tip_amount"),
    coalesce(col("tip_amount"), lit(0)).alias("tip_or_zero")
).show(3)

# 删除包含 NULL 的行
df.dropna(subset=["passenger_count", "trip_distance"]).count()

## 2.6 🟡 复杂数据类型 (Array, Map, Struct)

In [ ]:
from pyspark.sql.functions import array, explode, explode_outer, struct, create_map, map_keys, map_values

# ===== Array 类型 =====
df_with_array = df.select(
    "tpep_pickup_datetime",
    array(col("fare_amount"), col("tip_amount"), col("tolls_amount")).alias("fee_components")
)
df_with_array.show(3, truncate=False)

In [ ]:
# ===== explode: 展开数组 =====
df_with_array.select(
    "tpep_pickup_datetime",
    explode("fee_components").alias("fee")
).show(6)

In [ ]:
# ===== Struct 类型: 嵌套结构 =====
df_with_struct = df.select(
    "tpep_pickup_datetime",
    struct(
        col("PULocationID").alias("pickup"),
        col("DOLocationID").alias("dropoff")
    ).alias("locations"),
    struct(
        col("fare_amount"),
        col("tip_amount"),
        col("total_amount")
    ).alias("fares")
)

df_with_struct.show(3, truncate=False)
df_with_struct.printSchema()

In [ ]:
# ===== 访问嵌套字段 =====
df_with_struct.select(
    col("locations.pickup"),
    col("fares.total_amount")
).show(3)

---

# Part 3: 性能优化 (Senior DE 核心技能)

这部分是区分 Junior 和 Senior 的关键！

## 3.1 🟢 分区 (Partitioning)

In [ ]:
# ===== 查看当前分区数 =====
print(f"当前分区数: {df.rdd.getNumPartitions()}")

# ===== repartition: 增加分区 (触发 Shuffle) =====
df_more = df.repartition(100)
print(f"repartition 后: {df_more.rdd.getNumPartitions()}")

# ===== coalesce: 减少分区 (不触发 Shuffle，但可能不均匀) =====
df_less = df.coalesce(10)
print(f"coalesce 后: {df_less.rdd.getNumPartitions()}")

In [ ]:
# ===== 按列 repartition: Join 优化 =====
# 如果要按 PULocationID 做大量分析，按它分区
df_by_location = df.repartition(50, "PULocationID")

# 后续按 PULocationID 的 groupBy 不需要 shuffle
df_by_location.groupBy("PULocationID").count().explain()

In [ ]:
# ===== 写入时分区 =====
# 按日期分区存储 - 查询时可以跳过不相关的分区
# df.write.partitionBy("date").parquet("/path/to/output")

# 注意: 分区列的 distinct 值不宜过多 (通常 < 10000)
# 否则会产生大量小文件

## 3.2 🟢 缓存 (Cache / Persist)

In [ ]:
from pyspark import StorageLevel

# ===== 何时使用缓存 =====
# 1. DataFrame 被多次使用
# 2. DataFrame 计算成本高
# 3. DataFrame 比较小，能放入内存

# cache() = persist(StorageLevel.MEMORY_AND_DISK)
df_filtered = df.filter(col("total_amount") > 50)
df_filtered.cache()  # 标记为缓存

# 触发缓存 (需要一个 Action)
print(f"行数: {df_filtered.count()}")

# 后续操作会使用缓存
df_filtered.groupBy("payment_type").count().show()
df_filtered.agg(avg("total_amount")).show()

In [ ]:
# ===== 存储级别 =====
"""
StorageLevel.MEMORY_ONLY       - 只放内存，放不下丢弃
StorageLevel.MEMORY_AND_DISK   - 内存放不下写磁盘 (默认)
StorageLevel.MEMORY_ONLY_SER   - 序列化后放内存，省空间
StorageLevel.DISK_ONLY         - 只放磁盘
StorageLevel.OFF_HEAP          - 堆外内存
"""

# 内存紧张时可以用序列化存储
df_filtered.persist(StorageLevel.MEMORY_AND_DISK_SER)

In [ ]:
# ===== 清除缓存 =====
df_filtered.unpersist()

# 或清除所有缓存
# spark.catalog.clearCache()

## 3.3 🟢 Broadcast 变量

In [ ]:
# ===== Broadcast Join =====
# 小表 Join 大表时，广播小表避免 Shuffle

# 方式1: 自动广播 (小于 spark.sql.autoBroadcastJoinThreshold)
# 默认 10MB

# 方式2: 显式广播
from pyspark.sql.functions import broadcast

large_df = df
small_df = payment_types

# 强制广播 small_df
result = large_df.join(broadcast(small_df), "payment_type")
result.explain()

In [ ]:
# ===== Broadcast 变量 (用于 UDF) =====
# 当需要在 UDF 中使用查找表时

lookup_dict = {1: "Credit Card", 2: "Cash", 3: "No Charge", 4: "Dispute"}
broadcast_lookup = spark.sparkContext.broadcast(lookup_dict)

from pyspark.sql.functions import udf

@udf(StringType())
def get_payment_name(payment_type):
    return broadcast_lookup.value.get(payment_type, "Unknown")

df.select("payment_type", get_payment_name("payment_type").alias("payment_name")).show(5)

## 3.4 🟡 避免数据倾斜 (Data Skew)

In [ ]:
# ===== 检测数据倾斜 =====
# 查看 key 分布
df.groupBy("PULocationID").count().orderBy(col("count").desc()).show(10)

# 如果某些 key 的数量远超其他，就存在倾斜

In [ ]:
# ===== 解决方案1: 加盐 (Salting) =====
from pyspark.sql.functions import concat, lit, floor, rand

# 假设 location_id=132 (JFK) 数据量特别大
# 加一个随机后缀分散数据

SALT_BUCKETS = 10

df_salted = df.withColumn(
    "salted_key",
    concat(col("PULocationID"), lit("_"), floor(rand() * SALT_BUCKETS).cast("int"))
)

# 先按 salted_key 聚合
partial_agg = df_salted.groupBy("salted_key").agg(
    count("*").alias("partial_count"),
    sum("total_amount").alias("partial_sum")
)

# 提取原始 key，再聚合
from pyspark.sql.functions import split

final_agg = partial_agg \
    .withColumn("PULocationID", split(col("salted_key"), "_")[0].cast("int")) \
    .groupBy("PULocationID").agg(
        sum("partial_count").alias("total_count"),
        sum("partial_sum").alias("total_sum")
    )

final_agg.orderBy(col("total_count").desc()).show(5)

In [ ]:
# ===== 解决方案2: AQE 自动处理 (Spark 3.0+) =====
# 开启 AQE 后，Spark 会自动优化倾斜 Join
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Skew Join enabled: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")

## 3.5 🟡 UDF 优化

In [ ]:
# ===== 尽量避免 UDF，使用内置函数 =====
# 内置函数是用 Catalyst 优化器优化过的，性能远好于 UDF

# BAD: UDF
@udf(DoubleType())
def calculate_tip_pct_udf(tip, fare):
    if fare and fare > 0:
        return tip / fare * 100
    return 0.0

# GOOD: 内置函数
from pyspark.sql.functions import when, round

tip_pct_builtin = round(
    when(col("fare_amount") > 0, col("tip_amount") / col("fare_amount") * 100)
    .otherwise(0), 2
)

In [ ]:
# ===== 如果必须用 UDF，使用 Pandas UDF (向量化) =====
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(DoubleType())
def calculate_tip_pct_pandas(tip: pd.Series, fare: pd.Series) -> pd.Series:
    return (tip / fare * 100).fillna(0)

# Pandas UDF 批量处理，比普通 UDF 快很多
df.select(
    "tip_amount",
    "fare_amount",
    calculate_tip_pct_pandas(col("tip_amount"), col("fare_amount")).alias("tip_pct")
).show(5)

---

# Part 4: 数据读写

## 4.1 🟢 读取各种格式

In [ ]:
# ===== Parquet (推荐) =====
df_parquet = spark.read.parquet("/workspace/data/nyc_taxi_2023_01.parquet")

# 读取多个文件
# df = spark.read.parquet("/path/to/data/*.parquet")

# 读取分区数据
# df = spark.read.parquet("/path/to/data/")  # 自动识别分区

In [ ]:
# ===== CSV =====
# df_csv = spark.read \
#     .option("header", "true") \
#     .option("inferSchema", "true") \
#     .option("delimiter", ",") \
#     .option("quote", '"') \
#     .option("escape", '"') \
#     .option("multiLine", "true") \
#     .option("mode", "DROPMALFORMED") \
#     .csv("/path/to/data.csv")

# 指定 Schema (比 inferSchema 更快更准确)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

custom_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("value", DoubleType(), True)
])

# df_csv = spark.read.schema(custom_schema).csv("/path/to/data.csv")

In [ ]:
# ===== JSON =====
# df_json = spark.read \
#     .option("multiLine", "true") \
#     .json("/path/to/data.json")

# JSON Lines (每行一个 JSON)
# df_jsonl = spark.read.json("/path/to/data.jsonl")

In [ ]:
# ===== JDBC (数据库) =====
# jdbc_url = "jdbc:postgresql://localhost:5432/mydb"
# 
# df_jdbc = spark.read \
#     .format("jdbc") \
#     .option("url", jdbc_url) \
#     .option("dbtable", "my_table") \
#     .option("user", "username") \
#     .option("password", "password") \
#     .option("driver", "org.postgresql.Driver") \
#     .option("numPartitions", 10) \
#     .option("partitionColumn", "id") \
#     .option("lowerBound", 1) \
#     .option("upperBound", 1000000) \
#     .load()

## 4.2 🟢 写入数据

In [ ]:
# ===== 写入模式 =====
"""
mode="overwrite"  - 覆盖已存在的数据
mode="append"     - 追加到已存在的数据
mode="ignore"     - 如果存在则跳过
mode="error"      - 如果存在则报错 (默认)
"""

# ===== 写入 Parquet =====
# df.write \
#     .mode("overwrite") \
#     .parquet("/path/to/output")

# 按列分区存储
# df.write \
#     .mode("overwrite") \
#     .partitionBy("year", "month") \
#     .parquet("/path/to/output")

In [ ]:
# ===== 控制输出文件数量 =====
# 方法1: coalesce (减少文件数)
# df.coalesce(10).write.parquet("/path/to/output")

# 方法2: repartition (重新分区)
# df.repartition(10).write.parquet("/path/to/output")

# 方法3: maxRecordsPerFile
# df.write \
#     .option("maxRecordsPerFile", 100000) \
#     .parquet("/path/to/output")

---

# Part 5: Structured Streaming 基础

## 5.1 🟡 流处理基本概念

In [ ]:
# ===== Structured Streaming 基本模式 =====
# 把流数据当做无限增长的 DataFrame 处理

# 1. 定义数据源 (Source)
# 2. 定义转换 (Transformation)
# 3. 定义输出 (Sink)
# 4. 启动查询

# 示例: 从 Parquet 模拟流
# streaming_df = spark.readStream \
#     .schema(df.schema) \
#     .option("maxFilesPerTrigger", 1) \
#     .parquet("/path/to/data")

In [ ]:
# ===== 输出模式 =====
"""
outputMode="append"   - 只输出新行 (适合简单转换)
outputMode="complete" - 输出全部结果 (适合聚合)
outputMode="update"   - 只输出更新的行
"""

# ===== 触发模式 =====
"""
trigger(once=True)                    - 执行一次后停止 (批处理)
trigger(processingTime='10 seconds')  - 每 10 秒处理一次
trigger(continuous='1 second')        - 连续处理，低延迟
trigger(availableNow=True)            - 处理所有可用数据后停止
"""
print("见上方注释")

In [ ]:
# ===== 完整示例: 模拟流处理 =====
from pyspark.sql.functions import window

# 创建测试数据目录
# 实际使用时数据会持续写入这个目录

# streaming_df = spark.readStream \
#     .schema(df.schema) \
#     .option("maxFilesPerTrigger", 1) \
#     .parquet("/workspace/data/streaming_input")

# 定义聚合
# aggregated = streaming_df \
#     .withWatermark("tpep_pickup_datetime", "1 hour") \
#     .groupBy(
#         window(col("tpep_pickup_datetime"), "10 minutes"),
#         "PULocationID"
#     ) \
#     .agg(
#         count("*").alias("trip_count"),
#         sum("total_amount").alias("revenue")
#     )

# 启动查询
# query = aggregated.writeStream \
#     .outputMode("complete") \
#     .format("console") \
#     .trigger(processingTime='30 seconds') \
#     .start()

# query.awaitTermination()

---

# Part 6: 综合练习

In [ ]:
# ===== 🔥 综合练习: 完整的 ETL Pipeline =====

# 需求: 构建一个数据管道，生成每日运营报表
# 
# 步骤:
# 1. 读取原始数据
# 2. 数据清洗 (过滤异常值)
# 3. 特征工程 (添加派生列)
# 4. 多维度聚合
# 5. 输出报表
#
# 要求:
# - 使用 DataFrame API
# - 注意性能优化 (缓存、分区)
# - 代码结构清晰

# 你的代码:


In [ ]:
# ===== 参考实现 =====
from pyspark.sql.functions import (
    col, when, hour, dayofweek, date_format,
    count, sum, avg, round, percentile_approx
)

# 1. 读取数据
raw_df = spark.read.parquet("/workspace/data/nyc_taxi_2023_01.parquet")

# 2. 数据清洗
clean_df = raw_df.filter(
    (col("trip_distance") > 0) &
    (col("trip_distance") < 100) &  # 过滤异常距离
    (col("total_amount") > 0) &
    (col("total_amount") < 500) &  # 过滤异常金额
    (col("passenger_count") > 0) &
    (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))  # 时间合理
)

# 3. 特征工程
enriched_df = clean_df \
    .withColumn("date", date_format("tpep_pickup_datetime", "yyyy-MM-dd")) \
    .withColumn("hour", hour("tpep_pickup_datetime")) \
    .withColumn("day_of_week", dayofweek("tpep_pickup_datetime")) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
    .withColumn("time_period",
        when((col("hour") >= 7) & (col("hour") <= 9), "morning_rush")
        .when((col("hour") >= 17) & (col("hour") <= 19), "evening_rush")
        .when((col("hour") >= 22) | (col("hour") <= 5), "night")
        .otherwise("regular")
    ) \
    .withColumn("trip_duration_min",
        round((col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60, 2)
    )

# 缓存中间结果 (会被多次使用)
enriched_df.cache()
print(f"清洗后记录数: {enriched_df.count():,}")

# 4. 多维度聚合

# 4.1 每日总览
daily_summary = enriched_df.groupBy("date").agg(
    count("*").alias("total_trips"),
    round(sum("total_amount"), 2).alias("total_revenue"),
    round(avg("total_amount"), 2).alias("avg_fare"),
    round(avg("trip_distance"), 2).alias("avg_distance"),
    round(avg("trip_duration_min"), 2).alias("avg_duration_min"),
    round(percentile_approx("total_amount", 0.5), 2).alias("median_fare")
).orderBy("date")

print("\n=== 每日总览 ===")
daily_summary.show(5)

# 4.2 时段分析
period_analysis = enriched_df.groupBy("time_period").agg(
    count("*").alias("trips"),
    round(avg("total_amount"), 2).alias("avg_fare"),
    round(avg("tip_amount"), 2).alias("avg_tip"),
    round(sum(when(col("payment_type") == 1, 1).otherwise(0)) * 100 / count("*"), 1).alias("credit_card_pct")
).orderBy(col("trips").desc())

print("\n=== 时段分析 ===")
period_analysis.show()

# 4.3 热门地点 Top 10
top_locations = enriched_df.groupBy("PULocationID").agg(
    count("*").alias("trips"),
    round(sum("total_amount"), 2).alias("revenue")
).orderBy(col("revenue").desc()).limit(10)

print("\n=== 热门上车地点 Top 10 ===")
top_locations.show()

# 5. 清理缓存
enriched_df.unpersist()

---

# 总结: Senior DE Spark 技能清单

## 核心概念
| 技能 | 重要度 | 自评 |
|------|--------|------|
| SparkSession 配置 | 🟢 必须 | ☐ |
| Transformation vs Action | 🟢 必须 | ☐ |
| 执行计划 (Explain) | 🟢 必须 | ☐ |
| 惰性求值 (Lazy Evaluation) | 🟢 必须 | ☐ |

## DataFrame API
| 技能 | 重要度 | 自评 |
|------|--------|------|
| 基础操作 (select, filter, withColumn) | 🟢 必须 | ☐ |
| 聚合操作 (groupBy, agg, pivot) | 🟢 必须 | ☐ |
| 窗口函数 (Window) | 🟢 必须 | ☐ |
| Join 操作 | 🟢 必须 | ☐ |
| 类型转换和 NULL 处理 | 🟡 熟练 | ☐ |
| 复杂类型 (Array, Struct, Map) | 🟡 熟练 | ☐ |

## 性能优化
| 技能 | 重要度 | 自评 |
|------|--------|------|
| 分区策略 (repartition, coalesce) | 🟢 必须 | ☐ |
| 缓存 (cache, persist) | 🟢 必须 | ☐ |
| Broadcast Join | 🟢 必须 | ☐ |
| 数据倾斜处理 | 🟡 熟练 | ☐ |
| UDF 优化 / Pandas UDF | 🟡 熟练 | ☐ |
| AQE (自适应查询执行) | 🟡 熟练 | ☐ |

## 数据读写
| 技能 | 重要度 | 自评 |
|------|--------|------|
| Parquet 读写 | 🟢 必须 | ☐ |
| CSV/JSON 读写 | 🟡 熟练 | ☐ |
| JDBC 读写 | 🟡 熟练 | ☐ |
| 分区存储 | 🟡 熟练 | ☐ |

## Structured Streaming
| 技能 | 重要度 | 自评 |
|------|--------|------|
| 基本概念 | 🟡 熟练 | ☐ |
| 输出模式和触发器 | 🟡 熟练 | ☐ |
| Watermark | 🔵 了解 | ☐ |
| Kafka 集成 | 🔵 了解 | ☐ |

In [ ]:
# 清理
# spark.stop()